In [25]:
# Standard library imports
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed

# Numerical and data science libraries
import numpy as np
import pandas as pd

# Scikit-learn imports
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, QuantileTransformer, RobustScaler, PowerTransformer
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# XGBoost import (optional - will check availability)
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    warnings.warn("XGBoost not available. xgboost classifier will be disabled.")

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("All imports successful!")
print(f"XGBoost available: {XGBOOST_AVAILABLE}")

All imports successful!
XGBoost available: True


In [26]:
# =============================================================================
# Block 1: Global Configuration
# =============================================================================
# This cell defines all configurable parameters for the 8-fold CV pipeline.
# Modify these values to adjust pipeline behavior without changing code logic.

# -----------------------------------------------------------------------------
# Data Configuration
# -----------------------------------------------------------------------------
# BIN_WIDTHS: List of bin widths to process. Add more BWs as needed.
# Design: List type allows easy extension for additional bin widths.
BIN_WIDTHS = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

# -----------------------------------------------------------------------------
# Cross-Validation Configuration
# -----------------------------------------------------------------------------
# N_OUTER_FOLDS: Number of outer CV folds (8-fold CV)
N_OUTER_FOLDS = 8

# RANDOM_SEED: Default seed for inner CV folds and classifier initialization.
# Note: Outer CV splits are controlled by OUTER_RS_LIST, not RANDOM_SEED.
RANDOM_SEED = 42

# -----------------------------------------------------------------------------
# Scaling Configuration
# -----------------------------------------------------------------------------
SCALER_TYPE_OUTER = 'PowerTransformer'  # Options: 'StandardScaler', 'QuantileTransformer', 'RobustScaler', 'PowerTransformer'
SCALER_TYPE_INNER = 'PowerTransformer'

# -----------------------------------------------------------------------------
# Feature Selection Configuration (Mutual Information, multi-RS × fold)
# -----------------------------------------------------------------------------
# PEARSON_THRESHOLD: Correlation threshold for feature filtering
PEARSON_THRESHOLD = 0.7

# ORIGINAL_ONLY: If True, only use original_ features (no wavelet)
ORIGINAL_ONLY = True

# MI multi-RS inner CV: 16 RS × 5 folds = 80 accumulations per outer fold.
# Mirrors cv_pipeline.ipynb GA layout: run_ga_on_fold → FULL_RS[:] → inner CV.
# Each inner fold fits its own prefilter + scaler on 4/5 of train and
# accumulates the fold's normalized MI scores into the frequency counter
# (instead of GA's 1/5-fold holdout AUC used as weight).
MI_N_RS = 16
MI_N_FOLDS = 5
FULL_RS = list(range(MI_N_RS))  # 16 seeds, aligned with cv_pipeline.ipynb

# Feature Selection Grid
K_GRID = [5, 10, 15, 20, 30, 40, 50]

# -----------------------------------------------------------------------------
# Outer CV Multi-RS Configuration
# -----------------------------------------------------------------------------
# Multiple RS outer CV for stability estimation
OUTER_RS_LIST = [42, 123, 456] # , 67, 2005, 2026
RESULTS_FILE = None  # Will be set dynamically
SAVE_ARTIFACTS = True
ARTIFACTS_DIR = None  # Will be set dynamically

# -----------------------------------------------------------------------------
# Directory Paths
# -----------------------------------------------------------------------------
# Base directories for data and experiments
DATA_DIR = Path('/home/ser/pipeline/data')
WORKSPACE_DIR = Path('/home/ser/pipeline/workspace/mi_multirs_r0.7_PowerTransformer')

# Set RESULTS_FILE and ARTIFACTS_DIR
RESULTS_FILE = WORKSPACE_DIR / 'pipeline_results.csv'
ARTIFACTS_DIR = WORKSPACE_DIR / 'artifacts'

# -----------------------------------------------------------------------------
# Display Configuration
# -----------------------------------------------------------------------------
print("=" * 70)
print("Block 1: Global Configuration")
print("=" * 70)
print(f"BIN_WIDTHS:       {BIN_WIDTHS}")
print(f"N_OUTER_FOLDS:   {N_OUTER_FOLDS}")
print(f"RANDOM_SEED:     {RANDOM_SEED}")
print(f"SCALER_TYPE_OUTER(test 10 cases): {SCALER_TYPE_OUTER}")
print(f"SCALER_TYPE_INNER(for MI): {SCALER_TYPE_INNER}")
print(f"PEARSON_THRESHOLD: {PEARSON_THRESHOLD}")
print(f"ORIGINAL_ONLY:   {ORIGINAL_ONLY}")
print(f"MI_N_RS:         {MI_N_RS}")
print(f"MI_N_FOLDS:      {MI_N_FOLDS}")
print(f"FULL_RS:         {FULL_RS} ({len(FULL_RS)} seeds)")
print(f"K_GRID:          {K_GRID}")
print(f"OUTER_RS_LIST:   {OUTER_RS_LIST}")
print(f"WORKSPACE_DIR:   {WORKSPACE_DIR}")
print(f"RESULTS_FILE:    {RESULTS_FILE}")
print(f"ARTIFACTS_DIR:   {ARTIFACTS_DIR}")
print("=" * 70)
print("Configuration loaded successfully!")


Block 1: Global Configuration
BIN_WIDTHS:       [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
N_OUTER_FOLDS:   8
RANDOM_SEED:     42
SCALER_TYPE_OUTER(test 10 cases): PowerTransformer
SCALER_TYPE_INNER(for MI): PowerTransformer
PEARSON_THRESHOLD: 0.7
ORIGINAL_ONLY:   True
MI_N_RS:         16
MI_N_FOLDS:      5
FULL_RS:         [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] (16 seeds)
K_GRID:          [5, 10, 15, 20, 30, 40, 50]
OUTER_RS_LIST:   [42, 123, 456]
WORKSPACE_DIR:   /home/ser/pipeline/workspace/mi_multirs_r0.7_PowerTransformer
RESULTS_FILE:    /home/ser/pipeline/workspace/mi_multirs_r0.7_PowerTransformer/pipeline_results.csv
ARTIFACTS_DIR:   /home/ser/pipeline/workspace/mi_multirs_r0.7_PowerTransformer/artifacts
Configuration loaded successfully!


In [27]:
# =============================================================================
# Block 2: CSV Loader and Feature/Label Extraction Helpers
# =============================================================================
# This cell defines robust helper functions for data loading, column detection,
# and workspace directory management.

def resolve_binwidth_csv(bw: int) -> Path:
    """
    Looks under DATA_DIR for Cine_output_20260606_binWidth_{bw}.csv and returns the path.
    """
    return DATA_DIR / f"Cine_output_20260606_binWidth_{bw}.csv"

def find_case_id_column(df: pd.DataFrame) -> str:
    """
    Detects CaseNumber or CaseID column case-insensitively.
    """
    for col in df.columns:
        if col.lower() in ['casenumber', 'caseid']:
            return col
    raise ValueError("No CaseNumber or CaseID column found in DataFrame.")

def find_label_column(df: pd.DataFrame) -> str:
    """
    Detects Label column case-insensitively.
    """
    for col in df.columns:
        if col.lower() == 'label':
            return col
    raise ValueError("No Label column found in DataFrame.")

def load_feature_csv(csv_path: Path) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """
    Reads CSV with pandas, detects CaseNumber/CaseID and Label case-insensitively,
    and returns the original DataFrame, label Series, and feature column names.
    """
    df = pd.read_csv(csv_path)
    case_id_col = find_case_id_column(df)
    label_col = find_label_column(df)
    
    labels = df[label_col]
    feature_cols = [col for col in df.columns if col not in [case_id_col, label_col]]
    
    return df, labels, feature_cols

def filter_original_only(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter DataFrame to only keep original features (no wavelet).
    Original features have 'original' in their name but NOT 'wavelet'.
    Preserves non-feature columns (CaseNumber/CaseID, Label).
    """
    non_feature_cols = [c for c in df.columns if c.lower() in ['label', 'casenumber', 'caseid']]
    feature_cols = [c for c in df.columns if c.lower() not in ['label', 'casenumber', 'caseid']]
    # Original features: contain 'original' but NOT 'wavelet'
    original_cols = [c for c in feature_cols
                     if 'original' in c.lower() and 'wavelet' not in c.lower()]
    return df[non_feature_cols + original_cols]

def safe_save_csv(df: pd.DataFrame, path: Path) -> None:
    """
    Safely saves a DataFrame to the specified path, creating parent directories if needed.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

In [28]:
# =============================================================================
# Block 3: Preprocessing Foundation (Fit-Train-Only Filters & Scaler Factory)
# =============================================================================
# This cell defines the preprocessing filters and scaler factory that will be
# used by the MI feature selection and holdout evaluation. All operations are strictly
# fit on training data only to prevent data leakage.

def make_scaler(scaler_type: str):
    """
    Factory function to create a scaler based on the configured SCALER_TYPE.
    Returns StandardScaler or QuantileTransformer(output_distribution='normal').
    """
    if scaler_type == 'StandardScaler':
        return StandardScaler()
    elif scaler_type == 'QuantileTransformer':
        return QuantileTransformer(output_distribution='normal', random_state=42)
    elif scaler_type == 'RobustScaler':
        return RobustScaler()
    elif scaler_type == 'PowerTransformer':
        return PowerTransformer(method='yeo-johnson', standardize=True)
    else:
        raise ValueError(f"Unknown scaler type: {scaler_type}")
        
def pearson_correlation_filter(X: pd.DataFrame, threshold: float) -> list[str]:
    """
    Pearson correlation filter that keeps one feature from each highly correlated group.
    Matches the Kaggle notebook's logic but operates on and returns pandas DataFrames/lists.
    """
    feature_names = X.columns.tolist()
    if threshold >= 1.0 or X.shape[1] <= 1:
        return feature_names

    corr_matrix = np.abs(np.corrcoef(X.values, rowvar=False))
    n_features = X.shape[1]
    to_drop = set()
    for a in range(n_features):
        if a in to_drop:
            continue
        for b in range(a + 1, n_features):
            if b in to_drop:
                continue
            if corr_matrix[a, b] > threshold:
                to_drop.add(b)

    keep_features = [feature_names[i] for i in range(n_features) if i not in to_drop]
    return keep_features

def fit_train_only_prefilter(X_train: pd.DataFrame, y_train: pd.Series, threshold: float = PEARSON_THRESHOLD) -> list[str]:
    """
    Applies VarianceThreshold(0) and Pearson correlation filtering fit only on X_train.
    Returns the selected feature names, never using test data.
    """
    # 1. Apply VarianceThreshold(0) to remove constant features
    vt = VarianceThreshold(threshold=0.0)
    vt.fit(X_train)
    vt_mask = vt.get_support()
    
    # Filter X_train to non-constant features
    X_train_vt = X_train.loc[:, vt_mask]
    
    # 2. Apply Pearson correlation filtering
    selected_features = pearson_correlation_filter(X_train_vt, threshold)
    
    return selected_features

In [29]:
# =============================================================================
# Block 3: Classifier Factory + build_feature_frequency_df
# =============================================================================

def get_classifier(clf_type: str, random_state: int = 42, inner: bool = True):
    if clf_type == 'rf':
        return RandomForestClassifier(
            n_estimators=100 if inner else 500,
            max_depth=5 if inner else None,
            random_state=random_state, n_jobs=1
        )
    elif clf_type == 'svm':
        return SVC(C=1.0, kernel='rbf', probability=True, random_state=random_state)
    elif clf_type == 'xgboost':
        if not XGBOOST_AVAILABLE:
            raise RuntimeError("XGBoost is not available.")
        return xgb.XGBClassifier(
            n_estimators=100 if inner else 200,
            max_depth=5 if inner else 6,
            random_state=random_state, eval_metric='logloss', n_jobs=1
        )
    else:
        raise ValueError(f"Unsupported classifier type: {clf_type}")

def build_feature_frequency_df(frequency_counter, raw_count_counter=None):
    """
    Build a feature frequency DataFrame from a frequency counter.
    
    Parameters:
    -----------
    frequency_counter : dict
        Dict mapping feature names to MI scores (float).
        Higher score = higher rank.
    raw_count_counter : dict, optional
        Dict mapping feature names to raw integer counts (unweighted).
        If provided, an additional 'raw_count' column is added for comparison.
    
    Returns:
    --------
    pd.DataFrame
        Columns: feature, frequency, rank (, raw_count if raw_count_counter provided).
        Sorted by frequency descending, then feature name ascending.
    """
    sorted_items = sorted(frequency_counter.items(), key=lambda x: (-x[1], x[0]))
    records = [{'feature': f, 'frequency': freq, 'rank': i}
               for i, (f, freq) in enumerate(sorted_items, 1)]
    df = pd.DataFrame(records)

    if raw_count_counter is not None and not df.empty:
        df['raw_count'] = df['feature'].map(raw_count_counter).fillna(0).astype(int)

    return df

In [30]:
# =============================================================================
# Block 3: MI Feature Selection Engine (Multi-RS × Inner-Fold, MI-weighted)
# =============================================================================
# This replaces the GA feature selection with Mutual Information (filter-based)
# approach. Mirrors cv_pipeline.ipynb GA layout:
#
#   run_mi_on_fold()
#     └─ for rs in FULL_RS[:]           # 16 seeds, parallel via ProcessPoolExecutor
#         └─ _run_mi_single_rs()
#              └─ StratifiedKFold(MI_N_FOLDS)   # 5 folds
#                  ├─ fit_train_only_prefilter + make_scaler  (fit on 4/5 only)
#                  └─ mutual_info_classif on 4/5 train
#                       → normalized MI scores accumulated into frequency counter
#
# Difference vs GA: GA uses the AUC on the 1/5 held-out fold as weight; here we
# use the normalized MI score computed on the 4/5 training fold as weight.

from collections import Counter


def compute_mutual_info_scores(X_train, y_train, random_state=42):
    """Compute MI scores for all features against target."""
    try:
        mi_scores = mutual_info_classif(X_train, y_train, random_state=random_state)
        return np.asarray(mi_scores, dtype=float)
    except Exception as e:
        print(f"  [WARNING] MI computation failed (rs={random_state}): {type(e).__name__}: {e}")
        return np.ones(X_train.shape[1], dtype=float)


def _run_mi_single_rs(X_train, y_train, rs, n_folds):
    """
    Run all inner folds for a single random state.

    For each of the n_folds inner splits:
      - 4/5 of (X_train, y_train) is the inner-train slice
      - Fit VarianceThreshold + Pearson filter on that 4/5 (fit-on-train-only)
      - Fit scaler on that 4/5
      - Compute MI on the scaled 4/5
      - Return [(feature_name, raw_mi_score), ...] for this fold
      (raw MI score is forwarded unchanged — a fold with low total MI
      contributes ~0, which is the standard filter-FS aggregation.)

    The caller accumulates raw MI into the frequency counter and increments
    raw_count by 1 for every feature that survived prefiltering.

    Returns
    -------
    list[list[tuple[str, float]]]
        Outer list has n_folds entries; each inner list is
        [(feature_name, raw_mi_score), ...] for that fold.
    """
    fold_results = []
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=rs)

    for fold_idx, (train_idx, _val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr = X_train.iloc[train_idx]
        y_tr = y_train.iloc[train_idx]

        # Fit prefilter on this 4/5 only (no leakage from val)
        selected_features = fit_train_only_prefilter(X_tr, y_tr, threshold=PEARSON_THRESHOLD)
        if len(selected_features) < 2:
            selected_features = list(X_tr.columns[:2])
        X_filtered = X_tr[selected_features]

        # Fit scaler on this 4/5 only
        scaler = make_scaler(SCALER_TYPE_INNER)
        X_scaled_vals = scaler.fit_transform(X_filtered)
        X_scaled = pd.DataFrame(X_scaled_vals, columns=X_filtered.columns, index=X_filtered.index)

        # MI on the 4/5 training slice — use RAW MI scores as weights.
        # (Earlier version normalized scores to sum=1 per fold to mimic GA's
        # AUC weighting range; but AUC∈[0,1] and MI∈[0,∞) are different scales,
        # normalizing makes a noise fold with total MI≈0 still contribute a
        # uniform 1/N — masking the very signal MI is meant to convey.
        # Raw MI is the standard filter-FS aggregation: a fold with no
        # discriminative power contributes ~0, not 1/N.)
        mi_scores = compute_mutual_info_scores(X_scaled, y_tr, random_state=rs)

        fold_results.append(list(zip(selected_features, mi_scores.tolist())))

    return fold_results


def run_mi_on_fold(X_train, y_train,
                   n_rs=MI_N_RS, n_folds=MI_N_FOLDS, random_seed=RANDOM_SEED):
    """
    Runs MI feature selection across MI_N_RS random states × MI_N_FOLDS inner
    folds (= 80 accumulations per outer fold) and returns a feature-frequency
    DataFrame identical in shape to cv_pipeline.ipynb's run_ga_on_fold output.

    Weighting: each inner fold contributes its raw MI scores (no per-fold
    normalization). Aggregating 80 folds, a feature's `frequency`
    is the sum of its raw MI scores across all inner folds — informative,
    stable MI features score highest, and folds with no discriminative
    signal contribute ~0. `raw_count` counts how many inner folds the
    feature survived prefiltering in (typically all 80, since prefilter
    is per-fold).

    Parallelism: the MI_N_RS random states are run in a ProcessPoolExecutor
    (one process per RS), matching the GA version's parallelism boundary.

    Parameters
    ----------
    X_train, y_train : pd.DataFrame, pd.Series
        The outer-fold training data (70 cases at N_OUTER_FOLDS=8).
    n_rs : int
        Number of inner random states (default 16, aligned with FULL_RS).
    n_folds : int
        Number of inner CV folds per RS (default 5).
    random_seed : int
        Kept for API compatibility (inner seeds come from FULL_RS).

    Returns
    -------
    feature_frequency_df : pd.DataFrame
        Columns: feature, frequency, rank, raw_count.
        Sorted by frequency desc.
    """
    rs_list = FULL_RS[:n_rs]
    frequency_counter = Counter()
    raw_count_counter = Counter()

    n_workers = min(len(rs_list), os.cpu_count() or 1)

    if n_workers > 1:
        with ProcessPoolExecutor(max_workers=n_workers) as executor:
            future_to_rs = {
                executor.submit(_run_mi_single_rs, X_train, y_train, rs, n_folds): rs
                for rs in rs_list
            }
            for future in as_completed(future_to_rs):
                rs = future_to_rs[future]
                try:
                    fold_results = future.result()
                    for fold_features_scores in fold_results:
                        for feat, score in fold_features_scores:
                            frequency_counter[feat] += score
                            raw_count_counter[feat] += 1
                except Exception as e:
                    print(f"  WARNING: RS {rs} failed: {type(e).__name__}: {e}")
    else:
        # Serial fallback (e.g. when cpu_count == 1 or debugging)
        for rs in rs_list:
            fold_results = _run_mi_single_rs(X_train, y_train, rs, n_folds)
            for fold_features_scores in fold_results:
                for feat, score in fold_features_scores:
                    frequency_counter[feat] += score
                    raw_count_counter[feat] += 1

    feature_frequency_df = build_feature_frequency_df(frequency_counter, raw_count_counter)

    # Summary (same format as GA version)
    print("\n" + "=" * 60)
    print(f"MI Feature Selection — Multi-RS × Inner-Fold "
          f"({n_rs} RS × {n_folds} folds = {n_rs * n_folds} accumulations)")
    print("=" * 60)

    mi_top10 = sorted(frequency_counter.items(), key=lambda x: (-x[1], x[0]))[:10]
    print("\n[MI Score] Top-10 features:")
    for rank, (feat, score) in enumerate(mi_top10, 1):
        print(f"  {rank:2d}. {feat:<35s}  score = {score:.4f}")

    raw_top10 = sorted(raw_count_counter.items(), key=lambda x: (-x[1], x[0]))[:10]
    print("\n[Raw Count] Top-10 features:")
    for rank, (feat, count) in enumerate(raw_top10, 1):
        print(f"  {rank:2d}. {feat:<35s}  count = {count}")
    print("=" * 60)

    return feature_frequency_df


# Backward-compatible alias: any downstream code referencing the old name
# (e.g. quick scripts that exec-load this notebook) keeps working.
def run_mi_simple(X_train, y_train, **kwargs):
    """Deprecated alias for run_mi_on_fold."""
    return run_mi_on_fold(X_train, y_train, **kwargs)


In [31]:
# =============================================================================
# Block 4: Holdout Evaluation Helpers
# =============================================================================
# This cell defines the holdout evaluation helper functions that select Top-K
# features from the MI frequency ranking and expose a reusable classifier factory
# for final holdout evaluation.

def select_top_k_features(feature_frequency_df: pd.DataFrame, k: int) -> list[str]:
    """
    Selects the top K features from the MI frequency ranking DataFrame.
    The returned list of feature names is ordered by frequency rank.
    """
    # Sort by rank to ensure correct order, then take the first k rows
    sorted_df = feature_frequency_df.sort_values('rank')
    top_k_df = sorted_df.head(k)
    return top_k_df['feature'].tolist()

def get_holdout_classifier(clf_type: str, random_state: int = RANDOM_SEED):
    """
    Exposes a reusable classifier factory for final holdout evaluation.
    Supports 'svm', 'rf', and 'xgboost'.
    """
    return get_classifier(clf_type, inner=False)

In [32]:
# =============================================================================
# Block 4: K_GRID Holdout Evaluation Loop
# =============================================================================
# This cell defines the holdout evaluation loop over the Top-K feature counts
# grid (K_GRID) and a helper to build a tidy predictions DataFrame.

def evaluate_holdout_k_grid(X_train, y_train, X_test, feature_frequency_df, clf_type,
                             k_grid=None, scaler_type=SCALER_TYPE_OUTER, case_ids=None):
    """
    Loops over Top-K feature counts, trains final holdout models, computes AUC,
    and returns predictions/results.

    Parameters:
    -----------
    X_train : pd.DataFrame
        Training feature DataFrame.
    y_train : pd.Series or np.ndarray
        Training label Series or array.
    X_test : pd.DataFrame
        Test feature DataFrame (may contain Label column).
    feature_frequency_df : pd.DataFrame
        Ranked feature frequency DataFrame from MI.
    clf_type : str
        Classifier type ('svm', 'rf', 'xgboost').
    k_grid : list of int, optional
        Grid of Top-K feature counts to evaluate. Defaults to K_GRID.
    scaler_type : str, default=SCALER_TYPE_OUTER
        Scaler type to use.
    case_ids : pd.Series or list, optional
        Original CaseID values for test samples.

    Returns:
    --------
    list of dict
        List of result dictionaries containing k, auc, selected_features, y_true, y_score, case_index, case_id.
    """
    if k_grid is None:
        k_grid = K_GRID

    # Extract y_true from X_test if it contains the label column
    try:
        label_col = find_label_column(X_test)
        y_true = X_test[label_col].values
    except Exception:
        y_true = None

    results = []
    for k in k_grid:
        # Select Top-K features from feature_frequency_df
        selected_features = select_top_k_features(feature_frequency_df, k)

        # Filter both train/test to these Top-K features
        X_train_filtered = X_train[selected_features]
        X_test_filtered = X_test[selected_features]

        # Fit the configured scaler on train only and transform train/test
        scaler = make_scaler(scaler_type)

        X_train_scaled = scaler.fit_transform(X_train_filtered)
        X_test_scaled = scaler.transform(X_test_filtered)

        # Train get_holdout_classifier(clf_type) on train
        clf = get_holdout_classifier(clf_type)
        clf.fit(X_train_scaled, y_train)

        # Predict probabilities on test
        y_score = clf.predict_proba(X_test_scaled)[:, 1]

        # Compute ROC AUC with roc_auc_score
        if y_true is not None:
            try:
                auc = roc_auc_score(y_true, y_score)
            except ValueError:
                auc = 0.5
        else:
            auc = None

        results.append({
            'k': k,
            'auc': auc,
            'selected_features': selected_features,
            'y_true': y_true,
            'y_score': y_score,
            'case_index': X_test.index.tolist(),
            'case_id': [str(cid) for cid in case_ids] if case_ids is not None else X_test.index.tolist()
        })

    return results

In [33]:
# =============================================================================
# Block 5: Checkpoint Helpers and Missing-BW Skip Logic
# =============================================================================
# This cell defines checkpointing and missing-BW skip logic.
# It ensures that the pipeline can skip already completed folds or missing input files.

def bw_csv_exists(bw: int) -> bool:
    """
    Checks if the binwidth CSV file exists.
    """
    return resolve_binwidth_csv(bw).exists()

# -----------------------------------------------------------------------------
# Result file I/O functions for long-format checkpoint
# -----------------------------------------------------------------------------
def load_results_df() -> pd.DataFrame:
    """Load existing pipeline_results.csv or return empty DataFrame."""
    cols = ['bw', 'outer_rs', 'fold_idx', 'classifier', 'k', 'auc', 'timestamp']
    if not RESULTS_FILE.exists():
        return pd.DataFrame(columns=cols)
    try:
        df = pd.read_csv(RESULTS_FILE, dtype={'bw': int, 'outer_rs': int, 'fold_idx': int, 'k': int, 'auc': float})
        return df
    except Exception as e:
        print(f"[WARNING] Cannot read {RESULTS_FILE}: {e}, starting from empty results.")
        return pd.DataFrame(columns=cols)

def is_fold_done(results_df, bw, outer_rs, fold_idx, k_grid=None, classifiers=None):
    """Check if all (classifier, k) combinations for a fold are already computed."""
    if k_grid is None: k_grid = K_GRID
    if classifiers is None: classifiers = ['svm', 'rf', 'xgboost', 'softvote-3']
    if results_df.empty: return False
    subset = results_df[(results_df['bw'] == bw) & (results_df['outer_rs'] == outer_rs) & (results_df['fold_idx'] == fold_idx)]
    if subset.empty: return False
    found_clfs = set(subset['classifier'].unique())
    found_ks = set(subset['k'].unique())
    return set(classifiers).issubset(found_clfs) and set(k_grid).issubset(found_ks)

def append_fold_results(rows):
    """Append rows to pipeline_results.csv (header only on first write)."""
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    df_new = pd.DataFrame(rows)
    write_header = not RESULTS_FILE.exists()
    df_new.to_csv(RESULTS_FILE, mode='a', header=write_header, index=False)

# -----------------------------------------------------------------------------
# In-memory softvote computation
# -----------------------------------------------------------------------------
def compute_softvote_in_memory(predictions_by_clf, k_grid=None):
    """Compute softvote-3 AUC from in-memory predictions (no disk I/O)."""
    if k_grid is None: k_grid = K_GRID
    softvote_aucs = {}
    for k in k_grid:
        scores_list = []
        y_true = None
        for clf_name in ['svm', 'rf', 'xgboost']:
            if clf_name not in predictions_by_clf: continue
            matching = [r for r in predictions_by_clf[clf_name] if r['k'] == k]
            if not matching: continue
            res = matching[0]
            scores_list.append(res['y_score'])
            if y_true is None: y_true = res['y_true']
        if not scores_list or y_true is None:
            softvote_aucs[k] = 0.5
            continue
        avg_score = np.mean(scores_list, axis=0)
        try:
            softvote_aucs[k] = roc_auc_score(y_true, avg_score)
        except ValueError:
            softvote_aucs[k] = 0.5
    return softvote_aucs

# -----------------------------------------------------------------------------
# Summary functions for multi-RS results
# -----------------------------------------------------------------------------
def compute_pipeline_summary(results_df):
    """Compute mean/std AUC grouped by (bw, classifier, k)."""
    summary = results_df.groupby(['bw', 'classifier', 'k'])['auc'].agg(['mean', 'std', 'count']).reset_index()
    summary.columns = ['bw', 'classifier', 'k', 'mean_auc', 'std_auc', 'n_folds']
    return summary

def compute_variance_decomposition(results_df):
    """Decompose AUC variance into between-RS and within-RS components."""
    rs_means = results_df.groupby(['bw', 'outer_rs', 'classifier', 'k'])['auc'].mean().reset_index().rename(columns={'auc': 'mean_auc_per_rs'})
    between_rs = rs_means.groupby(['bw', 'classifier', 'k'])['mean_auc_per_rs'].std().reset_index().rename(columns={'mean_auc_per_rs': 'between_rs_std'})
    within_rs = results_df.groupby(['bw', 'outer_rs', 'classifier', 'k'])['auc'].std().reset_index().rename(columns={'auc': 'within_rs_std'})
    within_rs_avg = within_rs.groupby(['bw', 'classifier', 'k'])['within_rs_std'].mean().reset_index().rename(columns={'within_rs_std': 'within_rs_std_avg'})
    return pd.merge(between_rs, within_rs_avg, on=['bw', 'classifier', 'k'])

def print_summary(results_df, classifiers=None, top_k_values=None):
    """Pretty-print pipeline summary table."""
    if classifiers is None: classifiers = ['softvote-3', 'svm', 'rf', 'xgboost']
    if top_k_values is None: top_k_values = sorted(K_GRID)[:4]
    summary = compute_pipeline_summary(results_df)
    n_rs = results_df['outer_rs'].nunique()
    n_folds = results_df['fold_idx'].nunique()
    print(f"\n{'='*70}")
    print(f"Pipeline Summary  ({n_rs} outer RS × {n_folds} folds per RS)")
    print(f"{'='*70}")
    for bw in sorted(results_df['bw'].unique()):
        print(f"\nBW = {bw}:")
        sub = summary[(summary['bw'] == bw) & (summary['k'].isin(top_k_values))]
        for clf in classifiers:
            clf_sub = sub[sub['classifier'] == clf].sort_values('k')
            if clf_sub.empty: continue
            parts = [f"k={row['k']}: {row['mean_auc']:.4f}±{row['std_auc']:.4f}" for _, row in clf_sub.iterrows()]
            print(f"  {clf:<12s}: {' | '.join(parts)}")

In [34]:
# =============================================================================
# Block 2: Outer CV Generator
# =============================================================================
# This cell defines the OuterCVGenerator class which handles the generation
# and loading of the 8-fold outer cross-validation splits.

class OuterCVGenerator:
    @staticmethod
    def load_splits_in_memory(bw: int, outer_rs: int) -> dict:
        """Load all outer fold splits for a given bin width and random state into memory."""
        csv_path = resolve_binwidth_csv(bw)
        df, labels, feature_cols = load_feature_csv(csv_path)
        # [ORIGINAL_ONLY] Filter to original_ features only if enabled
        if ORIGINAL_ONLY:
            df = filter_original_only(df)
            feature_cols = [c for c in df.columns if c.lower() not in ['label', 'casenumber', 'caseid']]
        skf = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=outer_rs)
        splits = {}
        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df, labels)):
            train_df = df.iloc[train_idx].reset_index(drop=True)
            test_df = df.iloc[test_idx].reset_index(drop=True)
            case_id_col = find_case_id_column(df)
            label_col = find_label_column(df)
            X_train = train_df.drop(columns=[case_id_col, label_col])
            y_train = train_df[label_col].reset_index(drop=True)
            train_case_ids = train_df[case_id_col].reset_index(drop=True)
            X_test = test_df.drop(columns=[case_id_col, label_col])
            y_test = test_df[label_col].reset_index(drop=True)
            test_case_ids = test_df[case_id_col].reset_index(drop=True)
            splits[fold_idx] = (X_train, y_train, X_test, y_test, test_case_ids, train_case_ids)
        return splits

In [35]:
# =============================================================================
# Block 5: Main Execution Loop and Pipeline Orchestrator
# =============================================================================
# This cell defines the main pipeline execution loop and the single-fold runner.
# It supports checkpointing via long-format CSV, multi-RS outer CV, in-memory I/O.
#
# Per-fold MI feature selection now runs MI_N_RS (=16) random states × MI_N_FOLDS
# (=5) inner folds = 80 accumulations, mirroring cv_pipeline.ipynb's GA layout
# (but weighted by inner-fold MI score instead of inner-fold held-out AUC).

def run_single_fold(bw, outer_rs, fold_idx, X_train, y_train, X_test, y_test, test_case_ids, train_case_ids, results_df,
                    k_grid=None, n_rs=MI_N_RS, n_folds=MI_N_FOLDS, random_seed=RANDOM_SEED):
    """
    Runs the MI feature selection and holdout evaluation for a single fold.
    Uses in-memory data — no CSV I/O for splits or predictions.
    
    Parameters:
    -----------
    bw, outer_rs, fold_idx : int
        Bin width, outer random state, fold index
    X_train, y_train, X_test, y_test : array-like
        Fold data (already split)
    test_case_ids, train_case_ids : pd.Series
        Case ID arrays
    results_df : pd.DataFrame
        Current results for checkpoint checking
    k_grid : list
        Top-K feature counts to evaluate
    n_rs : int
        Number of random states for MI (default MI_N_RS=16)
    n_folds : int
        Number of inner folds per RS for MI (default MI_N_FOLDS=5)
    random_seed : int
        Base random seed (kept for API compatibility)
        
    Returns:
    --------
    list of dict
        Rows appended to results_df (empty if skipped)
    """
    if k_grid is None: k_grid = K_GRID
    
    # Checkpoint: skip if all (classifier, k) combos already done
    if is_fold_done(results_df, bw, outer_rs, fold_idx, k_grid=k_grid):
        print(f"[SKIP] BW={bw}, RS={outer_rs}, fold={fold_idx} — already done.")
        return []
    
    # Prepare X_test for holdout evaluation (needs Label column)
    X_test_eval = X_test.copy()
    X_test_eval['Label'] = y_test.values if hasattr(y_test, 'values') else y_test
    
    # Run MI on fold (16 RS × 5 inner folds = 80 accumulations)
    print(f"Running MI feature selection on fold {fold_idx} "
          f"(n_rs={n_rs}, n_folds={n_folds}, accumulations={n_rs * n_folds})...")
    feature_frequency_df = run_mi_on_fold(
        X_train, y_train,
        n_rs=n_rs, n_folds=n_folds, random_seed=random_seed
    )
    
    # Save MI artifacts if enabled
    if SAVE_ARTIFACTS:
        art_dir = ARTIFACTS_DIR / f"BW_{bw}" / f"outer_rs_{outer_rs}" / f"fold_{fold_idx}"
        art_dir.mkdir(parents=True, exist_ok=True)
        freq_path = art_dir / 'feature_frequencies.csv'
        safe_save_csv(feature_frequency_df, freq_path)
        print(f"  Saved feature_frequencies.csv → {art_dir}")
    
    # Evaluate holdout for each classifier in-memory
    predictions_by_clf = {}
    for clf in ['svm', 'rf', 'xgboost']:
        print(f"  Evaluating {clf}...")
        ho_results = evaluate_holdout_k_grid(
            X_train, y_train, X_test_eval,
            feature_frequency_df, clf,
            k_grid=k_grid, scaler_type=SCALER_TYPE_OUTER,
            case_ids=test_case_ids
        )
        predictions_by_clf[clf] = ho_results
    
    # Compute softvote in-memory
    softvote_aucs = compute_softvote_in_memory(predictions_by_clf, k_grid)
    
    # Assemble result rows
    rows = []
    timestamp = datetime.now().isoformat()
    for clf in ['svm', 'rf', 'xgboost', 'softvote-3']:
        for k in k_grid:
            if clf == 'softvote-3':
                auc_val = softvote_aucs.get(k, 0.5)
            else:
                matching = [r for r in predictions_by_clf[clf] if r['k'] == k]
                auc_val = matching[0]['auc'] if matching else 0.5
            rows.append({
                'bw': bw,
                'outer_rs': outer_rs,
                'fold_idx': fold_idx,
                'classifier': clf,
                'k': k,
                'auc': auc_val,
                'timestamp': timestamp
            })
    
    # Persist to long-format checkpoint
    append_fold_results(rows)
    
    return rows

def run_pipeline(force_recompute=False, bin_widths=None, outer_rs_list=None, n_outer_folds=None,
                 n_rs=MI_N_RS, n_folds=MI_N_FOLDS, random_seed=RANDOM_SEED, k_grid=None):
    """
    Main pipeline execution loop — multi-RS outer CV with in-memory I/O.
    
    This is the MI-based version of the pipeline. Instead of running GA for feature
    selection, it uses Mutual Information to rank features. Per outer fold, MI is
    run with n_rs (=16) random states × n_folds (=5) inner folds.
    
    Parameters:
    -----------
    force_recompute : bool
        If True, backup existing RESULTS_FILE and start fresh
    bin_widths : list of int
        Bin widths to process
    outer_rs_list : list of int
        Random states for outer CV splits
    n_outer_folds : int
        Number of outer folds per RS
    n_rs : int
        Number of random states for MI (default MI_N_RS=16)
    n_folds : int
        Number of inner folds per RS for MI (default MI_N_FOLDS=5)
    random_seed : int
        Base random seed (kept for API compatibility)
    k_grid : list
        Top-K feature counts to evaluate
    """
    # Initialize configuration
    if bin_widths is None:
        bin_widths = BIN_WIDTHS
    if outer_rs_list is None:
        outer_rs_list = OUTER_RS_LIST
    if n_outer_folds is None:
        n_outer_folds = N_OUTER_FOLDS
    if k_grid is None:
        k_grid = K_GRID
    
    # Load existing results for checkpointing
    results_df = load_results_df()
    
    # Backup existing results if force_recompute
    if force_recompute and RESULTS_FILE.exists():
        backup_path = RESULTS_FILE.with_suffix('.backup.csv')
        import shutil
        shutil.copy(RESULTS_FILE, backup_path)
        print(f"Backed up existing results to: {backup_path}")
        results_df = pd.DataFrame(columns=['bw', 'outer_rs', 'fold_idx', 'classifier', 'k', 'auc', 'timestamp'])
    
    # Main execution loop
    total_folds = len(bin_widths) * len(outer_rs_list) * n_outer_folds
    processed_folds = 0
    
    for bw in bin_widths:
        if not bw_csv_exists(bw):
            print(f"[WARNING] BW={bw} CSV not found, skipping.")
            continue
            
        for outer_rs in outer_rs_list:
            print(f"\n{'='*70}")
            print(f"Processing BW={bw}, Outer RS={outer_rs}")
            print(f"{'='*70}")
            
            # Load all splits for this BW and RS into memory
            splits = OuterCVGenerator.load_splits_in_memory(bw, outer_rs)
            
            for fold_idx in range(n_outer_folds):
                if fold_idx not in splits:
                    print(f"[WARNING] Fold {fold_idx} not found in splits for BW={bw}, RS={outer_rs}")
                    continue
                    
                X_train, y_train, X_test, y_test, test_case_ids, train_case_ids = splits[fold_idx]
                
                print(f"\n--- Fold {fold_idx} (train: {len(X_train)}, test: {len(X_test)}) ---")
                
                # Run single fold
                rows = run_single_fold(
                    bw, outer_rs, fold_idx,
                    X_train, y_train, X_test, y_test, test_case_ids, train_case_ids,
                    results_df, k_grid=k_grid,
                    n_rs=n_rs, n_folds=n_folds, random_seed=random_seed
                )
                
                if rows:
                    # Update results_df with new rows
                    new_rows_df = pd.DataFrame(rows)
                    results_df = pd.concat([results_df, new_rows_df], ignore_index=True)
                    processed_folds += 1
    
    # Print final summary
    print(f"\n{'='*70}")
    print(f"Pipeline completed!")
    print(f"Processed {processed_folds}/{total_folds} folds")
    print(f"Results saved to: {RESULTS_FILE}")
    print(f"{'='*70}")
    
    # Print summary statistics
    if not results_df.empty:
        print_summary(results_df)
    
    return results_df


In [36]:
# =============================================================================
# Entry Point
# =============================================================================

if __name__ == "__main__":
    print("Starting MI-based CV pipeline (multi-RS × inner-fold, MI-weighted)...")
    print(f"Configuration:")
    print(f"  BIN_WIDTHS: {BIN_WIDTHS}")
    print(f"  OUTER_RS_LIST: {OUTER_RS_LIST}")
    print(f"  N_OUTER_FOLDS: {N_OUTER_FOLDS}")
    print(f"  MI_N_RS: {MI_N_RS}")
    print(f"  MI_N_FOLDS: {MI_N_FOLDS}")
    print(f"  FULL_RS: {FULL_RS}")
    print(f"  accumulations per outer fold: {MI_N_RS * MI_N_FOLDS}")
    print(f"  K_GRID: {K_GRID}")
    print(f"  RESULTS_FILE: {RESULTS_FILE}")
    
    # Run the pipeline
    results = run_pipeline(force_recompute=False)
    
    print("\nPipeline execution completed successfully!")

Starting MI-based CV pipeline (multi-RS × inner-fold, MI-weighted)...
Configuration:
  BIN_WIDTHS: [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
  OUTER_RS_LIST: [42, 123, 456]
  N_OUTER_FOLDS: 8
  MI_N_RS: 16
  MI_N_FOLDS: 5
  FULL_RS: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  accumulations per outer fold: 80
  K_GRID: [5, 10, 15, 20, 30, 40, 50]
  RESULTS_FILE: /home/ser/pipeline/workspace/mi_multirs_r0.7_PowerTransformer/pipeline_results.csv

Processing BW=5, Outer RS=42



--- Fold 0 (train: 70, test: 10) ---
Running MI feature selection on fold 0 (n_rs=16, n_folds=5, accumulations=80)...

MI Feature Selection — Multi-RS × Inner-Fold (16 RS × 5 folds = 80 accumulations)

[MI Score] Top-10 features:
   1. Cine_3D_ES_original_shape_MajorAxisLength  score = 9.8953
   2. Cine_3D_ES_original_gldm_LargeDependenceLowGrayLevelEmphasis  score = 9.0711
   3. Cine_3D_ES_original_shape_Elongation  score = 7.5366
   4. Cine_3D_ES_original_shape_Maximum2DDiameterSlice  score = 7.4917
   5. Cine_3D_ES_original_shape_Flatness   score = 7.0932
   6. Cine_3D_ES_original_shape_Sphericity  score = 4.6851
   7. Cine_3D_ED_original_shape_Sphericity  score = 4.2522
   8. Cine_3D_ES_original_gldm_GrayLevelNonUniformity  score = 3.4382
   9. Cine_3D_ED_original_ngtdm_Strength   score = 3.1526
  10. Cine_3D_ES_original_firstorder_Kurtosis  score = 2.4690

[Raw Count] Top-10 features:
   1. Cine_3D_ED_original_shape_Elongation  count = 80
   2. Cine_3D_ES_original_firstorder_10Pe